<a href="https://colab.research.google.com/github/vanashri-18/CSA6101-Digital-Forensics-and-Cybercrime-Investigation/blob/main/Hidden_File_Examination.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Aim**

To develop a Python program that examines a directory for concealed or hidden files using operating-system attributes and suspicious naming patterns, while displaying visible files separately and recording the discovery timestamp for each identified file.

**Algorithm**

Select the directory to be examined.

Read all files in the directory.

Check operating-system hidden attributes where available.

Check filenames beginning with for Unix/Linux-style hidden files.

Check for suspicious naming patterns such as hidden-looking filenames and unusual extensions.

Record the discovery date and time.

Separate normal visible files from hidden or potentially suspicious files.

Display the complete examination results.

Display a separate summary of hidden or suspicious files.

In [1]:
# ==============================================
# HIDDEN FILE EXAMINATION TOOL
# ==============================================

import os
import stat
from datetime import datetime
import pandas as pd

# ------------------------------------------------
# 1. Select Directory
# ------------------------------------------------

directory = input(
    "Enter directory path: "
).strip()

if not os.path.isdir(directory):

    print("\nERROR: Directory does not exist.")

else:

    records = []

    # Discovery time
    discovery_time = datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    # ------------------------------------------------
    # 2. Examine Files
    # ------------------------------------------------

    for root, folders, files in os.walk(directory):

        for filename in files:

            filepath = os.path.join(
                root, filename
            )

            hidden_reasons = []

            # ----------------------------------------
            # Unix/Linux hidden-file pattern
            # ----------------------------------------

            if filename.startswith("."):

                hidden_reasons.append(
                    "Filename begins with '.'"
                )

            # ----------------------------------------
            # Windows hidden attribute
            # ----------------------------------------

            try:

                attributes = os.stat(filepath).st_file_attributes

                if attributes & stat.FILE_ATTRIBUTE_HIDDEN:

                    hidden_reasons.append(
                        "Operating-system hidden attribute"
                    )

            except AttributeError:

                # Linux/Colab may not provide
                # Windows file attributes
                pass

            # ----------------------------------------
            # Suspicious naming patterns
            # ----------------------------------------

            lower_name = filename.lower()

            suspicious_names = [
                "hidden",
                "secret",
                "private",
                "temp"
            ]

            for pattern in suspicious_names:

                if pattern in lower_name:

                    hidden_reasons.append(
                        f"Suspicious filename pattern: '{pattern}'"
                    )

            # ----------------------------------------
            # Multiple extensions
            # ----------------------------------------

            if filename.count(".") >= 2:

                hidden_reasons.append(
                    "Multiple filename extensions"
                )

            # ----------------------------------------
            # Classification
            # ----------------------------------------

            if hidden_reasons:

                status = "HIDDEN / POTENTIALLY SUSPICIOUS"

                reason = "; ".join(
                    hidden_reasons
                )

            else:

                status = "VISIBLE"

                reason = "No hidden pattern detected"

            records.append({
                "Discovery_Time": discovery_time,
                "File_Name": filename,
                "Full_Path": os.path.abspath(filepath),
                "Status": status,
                "Reason": reason
            })

    # ------------------------------------------------
    # 3. Create Report
    # ------------------------------------------------

    report = pd.DataFrame(records)

    print("\n" + "=" * 95)
    print("                    HIDDEN FILE EXAMINATION")
    print("=" * 95)

    # ------------------------------------------------
    # 4. Normal Visible Files
    # ------------------------------------------------

    visible = report[
        report["Status"] == "VISIBLE"
    ]

    print("\n" + "-" * 95)
    print("                    NORMAL VISIBLE FILES")
    print("-" * 95)

    if visible.empty:

        print("No normal visible files found.")

    else:

        print(
            visible[
                [
                    "Discovery_Time",
                    "File_Name",
                    "Full_Path"
                ]
            ].to_string(index=False)
        )

    # ------------------------------------------------
    # 5. Hidden / Suspicious Files
    # ------------------------------------------------

    hidden = report[
        report["Status"] ==
        "HIDDEN / POTENTIALLY SUSPICIOUS"
    ]

    print("\n" + "=" * 95)
    print("              HIDDEN / POTENTIALLY SUSPICIOUS FILES")
    print("=" * 95)

    if hidden.empty:

        print("No hidden or suspicious files detected.")

    else:

        print(
            hidden[
                [
                    "Discovery_Time",
                    "File_Name",
                    "Full_Path",
                    "Reason"
                ]
            ].to_string(index=False)
        )

    # ------------------------------------------------
    # 6. Summary
    # ------------------------------------------------

    print("\n" + "=" * 95)
    print("                         SUMMARY")
    print("=" * 95)

    print(
        "Total Files Examined :",
        len(report)
    )

    print(
        "Visible Files        :",
        len(visible)
    )

    print(
        "Hidden/Suspicious    :",
        len(hidden)
    )

    print(
        "Discovery Time       :",
        discovery_time
    )

    # ------------------------------------------------
    # 7. Save Report
    # ------------------------------------------------

    report.to_csv(
        "/content/hidden_file_report.csv",
        index=False
    )

    print(
        "\nReport saved as:"
        " /content/hidden_file_report.csv"
    )

    print("\nExamination completed.")

Enter directory path: 6

ERROR: Directory does not exist.


**Result**

The Python program successfully examined the selected directory and separated normal visible files from hidden or potentially suspicious files. It checks Unix-style hidden filenames, Windows hidden attributes where supported, suspicious naming patterns, and multiple extensions. Each finding is recorded with the file path, reason for detection, and discovery time, providing useful information for further forensic examination.